In [12]:
!free -m


              total        used        free      shared  buff/cache   available
Mem:         254917       36054      197023           0       21839      216707
Swap:             0           0           0


In [13]:
import pandas as pd
import os
import glob
import numpy as np

# Specify the path to your folder containing CSV files
folder_path = 'StudentProficiencyData'

# Use glob to get a list of all CSV files in the folder
file_pattern = os.path.join(folder_path, '*.csv.gz')
file_list = glob.glob(file_pattern)

# Initialize an empty DataFrame to concatenate the individual files
df_list = []

column_names = ['studentid', 'previousitems', 'itemdifficulty', 'currentperformance', 'currentscore', 'itemid', 'eventtime', 'skillid', 'subskillid']

# Loop through each file and load it into a DataFrame with the specified column names and data types
for file in file_list:
    df = pd.read_csv(file, compression='gzip', names=column_names, parse_dates=['eventtime'], na_values=['NA'])
    
    # Explicitly convert 'previousitems' column to integer
    #df['previousitems'] = pd.to_numeric(df['previousitems'], errors='coerce', downcast='integer')
    #df['currentscore'] = pd.to_numeric(df['currentscore'], errors='coerce', downcast='float')
    
    df_list.append(df)
    
# Concatenate all DataFrames in the list into a single DataFrame
final_df = pd.concat(df_list, ignore_index=True)

# Now, final_df contains the data from all CSV files

import pandas as pd

# Assuming your DataFrame is named 'df'
# Sort the DataFrame by 'eventtime' in descending order
final_df = final_df.sort_values(by='eventtime', ascending=False)

# Group by 'studentid' and 'skillid', then get the last row for each group
result_df = final_df.groupby(['studentid', 'skillid']).head(1)

# Reset index if needed
result_df = result_df.reset_index(drop=True)

print(np.shape(final_df))
print(np.shape(result_df))
result_df


/tmp/ipykernel_21766/3637138083.py:20: DtypeWarning: Columns (7,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, compression='gzip', names=column_names, parse_dates=['eventtime'], na_values=['NA'])
/tmp/ipykernel_21766/3637138083.py:20: DtypeWarning: Columns (7,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, compression='gzip', names=column_names, parse_dates=['eventtime'], na_values=['NA'])
/tmp/ipykernel_21766/3637138083.py:20: DtypeWarning: Columns (7,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, compression='gzip', names=column_names, parse_dates=['eventtime'], na_values=['NA'])
/tmp/ipykernel_21766/3637138083.py:20: DtypeWarning: Columns (7,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, compression='gzip', names=column_names, parse_dates=['eventtime'], na_values=

(37400000, 9)
(21961937, 9)


,studentid,previousitems,itemdifficulty,currentperformance,currentscore,itemid,eventtime,skillid,subskillid
0,fancy-api@1,100,proficient,\\N,5,6,\\N,\\N,\\N
1,progress@1,100,\\N,0.168833,\\N,\\N,999,0.998467,bad4fe019982
2,progress@1,100,\\N,0.306056,\\N,\\N,999,0.79760008,564fbb175d08
3,progress@1,100,\\N,0.027347,\\N,\\N,999,0.925915675,e636c7fc8a19
4,progress@1,100,\\N,0.345079,\\N,\\N,999,1.14757662,5ccc89efa3f4
...,...,...,...,...,...,...,...,...,...
21961932,progress@1,0,\\N,-0.12559776,\\N,\\N,10,0.04710289,d6789fede606
21961933,progress@1,100,\\N,0.026273,\\N,\\N,10,0.960531178,16ea56d8c39b
21961934,progress@1,0,\\N,-0.24984412,\\N,\\N,10,0.40417532,b177f659bcbb
21961935,progress@1,100,\\N,0.115142,\\N,\\N,10,1.02777605,8c121a213482


In [14]:
final_df=result_df
df=result_df

In [15]:
# Sort DataFrame by eventtime
final_df.sort_values(['eventtime', 'studentid', 'skillid'], inplace=True)
final_df


,studentid,previousitems,itemdifficulty,currentperformance,currentscore,itemid,eventtime,skillid,subskillid
21961936,progress@1,0,\\N,-0.498608,\\N,\\N,1,\\N,51b9e59d9fa5
21556480,progress@1,100,\\N,-0.41975808,\\N,\\N,10,-0.0001027,61361cb6f342
21835976,progress@1,50,\\N,-0.02250557,\\N,\\N,10,-0.000128756,9d8b62acd6c5
21672478,progress@1,0,\\N,-0.645272,\\N,\\N,10,-0.00013175,4b3f33d35a5e
21910534,progress@1,0,\\N,-0.180555,\\N,\\N,10,-0.00014694,45d4723e3d1a
...,...,...,...,...,...,...,...,...,...
4,progress@1,100,\\N,0.345079,\\N,\\N,999,1.14757662,5ccc89efa3f4
112,progress@1,100,\\N,0.216641,\\N,\\N,999,1.15143615,56d75af3d97c
152,progress@1,100,\\N,0.381178,\\N,\\N,999,1.19663164,359c2cff99f8
130,progress@1,100,\\N,0.293124,\\N,\\N,999,1.21381736,f19ba65630d3


In [16]:
# Get the last entry for each combination of student and skill
last_entry_df = df.groupby(['studentid', 'skillid']).last().reset_index()

# Pivot the tables
score_table = last_entry_df.pivot(index='studentid', columns='skillid', values='currentscore')
timestamps = last_entry_df.pivot(index='studentid', columns='skillid', values='eventtime')

In [17]:
np.shape(score_table)

(2, 21961936)

In [18]:
!free -m


              total        used        free      shared  buff/cache   available
Mem:         254917       55176      177840           0       21900      197670
Swap:             0           0           0


In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

def loss_function(params, R, num_users, num_items, num_factors, lambda_reg):
    U = params[:num_users * num_factors].reshape((num_users, num_factors))
    V = params[num_users * num_factors:].reshape((num_items, num_factors))

    predicted_scores = np.dot(U, V.T)

    # Calculate reconstruction loss (MSE)
    reconstruction_loss = np.mean((R - predicted_scores) ** 2)

    # Calculate regularization loss for user and item matrices
    regularization_loss = lambda_reg * (np.sum(U**2) + np.sum(V**2))

    # Total loss (include regularization if needed)
    total_loss = reconstruction_loss + regularization_loss

    return total_loss

##MEMORY ERROR SO LETS SUBSAMPLE (& ENSURE COVERAGE OF ALL SKILLIDS)
def subsample_data(score_table, num_users_to_sample):
    # Get unique skill IDs
    unique_skill_ids = score_table.columns.unique()

    # Initialize an empty list to store sampled users
    sampled_users = []

    # Track covered skill IDs
    covered_skill_ids = set()

    # Sample users until the desired number is reached and all skill IDs are covered
    while len(sampled_users) < num_users_to_sample:
        # Randomly select a user
        user = np.random.choice(score_table.index.unique())

        # Check if the sampled user covers any new skill IDs
        new_covered_skill_ids = set(score_table.loc[user].dropna().index) - covered_skill_ids

        # If new skill IDs are covered, add the user to the sampled list
        if new_covered_skill_ids:
            sampled_users.append(user)
            covered_skill_ids.update(new_covered_skill_ids)

    # Subsample score_table based on selected users
    sampled_score_table = score_table[score_table.index.isin(sampled_users)]

    return sampled_score_table

# Subsample data
num_users_to_sample = 10000  # Adjust as needed
sampled_score_table, _ = subsample_data(score_table, num_users_to_sample)

# Initialize variables
num_users, num_items = sampled_score_table.shape
num_factors = 100  # Adjust as needed
lambda_reg = 0.01  # Regularization parameter

# Flatten user and item matrices for optimization
initial_params = np.random.rand(num_users * num_factors + num_items * num_factors)

# Minimize the loss function
result = minimize(loss_function, initial_params, args=(sampled_score_table, num_users, num_items, num_factors, lambda_reg))

# Extract learned matrices
U_final = result.x[:num_users * num_factors].reshape((num_users, num_factors))
V_final = result.x[num_users * num_factors:].reshape((num_items, num_factors))

# Make predictions for new items
new_item_predictions = np.dot(U_final, V_final.T)



In [ ]:
# Evaluate the model
mse = mean_squared_error(sampled_score_table, np.dot(U_final, V_final.T))
print(f'Mean Squared Error: {mse}')


In [ ]:
np.shape(sampled_score_table), np.shape(new_item_predictions), np.shape(result), num_users, num_items, np.shape(V_final), np.shape(U_final)